# demo_bench

> Phase 8 benchmark: the main grid with the pre-registered confirmatory tests, scaling curves and secondary metrics; side sweeps and studies A (team-optimal exploration, reward schemes) and B (persistence and credit).

Spec: `specs/multiagent/phase8/` (design-low-level section 7). Artifacts under `results/multiagent/bench/{quick,full,sweeps}/` (gitignored): `runs.csv`, `metrics.json`, `report.md` and figures. Normal tests run the quick grid (pilot seeds, budgets / 10, plus one cell per sweep path); `--flags slow` runs the main grid on the confirmatory seeds; `--flags sweeps` runs the side sweeps and study B.

In [ ]:
import os
import subprocess
import time

import matplotlib.pyplot as plt

from explRL.bench_figures import mainFigures, sweepFigures
from explRL.bench_grid import PREREG, preregHash, mainGrid, sideSweeps, runGrid, writeCSV, confirm
from explRL.bench_report import slopes, sweepTable, teamOptimal, studyB, report
from explRL.train import demo_artifact_dir, save_demo_figure, save_demo_metrics, save_demo_report

DN = 'multiagent/bench'
MODE = 'quick'
WORKERS = min(8, os.cpu_count() or 1)
DEVIATIONS = 'none'


def gitCommit():
    try:
        out = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True)
        return out.stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def jsonTable(tab):
    "Side-sweep table with string keys, for metrics.json."
    return {f: {l: {f'{c}/{n}': v for (c, n), v in cols.items()} for l, cols in lv.items()} for f, lv in tab.items()}


def save(mode, rows, figs, meta, extra, text):
    writeCSV(rows, demo_artifact_dir(mode, DN) / 'runs.csv')
    for name, f in figs.items():
        save_demo_figure(name, fig=f, mode=mode, demo_name=DN)
    plt.close('all')
    save_demo_metrics({**meta, **extra}, mode=mode, demo_name=DN)
    return save_demo_report(text, mode=mode, demo_name=DN)

In [ ]:
#| slow
MODE = 'full'

## Main grid and confirmatory tests

In [ ]:
t0 = time.time()
cells = mainGrid(MODE)
rows = runGrid(cells, WORKERS)
if MODE == 'quick':                      # the quick run also exercises every side-sweep and study B path
    cells = cells + sideSweeps('quick')
    rows = rows + runGrid(sideSweeps('quick'), WORKERS)
wall = time.time() - t0
conf = confirm(rows, seeds=(1000, 1001) if MODE == 'quick' else None)
sl = slopes(rows)
tab = sweepTable(rows)
opt, sb = teamOptimal(tab), studyB(rows)
meta = dict(mode=MODE, cells=len(cells), workers=WORKERS, wallSeconds=wall, preregHash=preregHash(),
            gitCommit=gitCommit(), prereg=PREREG, deviations=DEVIATIONS)
path = save(MODE, rows, {**mainFigures(rows, conf, PREREG), **sweepFigures(tab, opt, sb)}, meta,
            dict(confirmatory=conf, scaling=sl, sweeps=jsonTable(tab), teamOptimalCUCB=opt, studyB=sb),
            report(meta, conf, sl, tab, opt, sb))
print(f'{MODE}: {len(rows)} rows in {wall:.0f} s ->', path)
for k, c in conf.items():
    print(k, c['verdict'], round(c['mean'], 4), [round(x, 4) for x in c['ci']], 'Holm p', round(c['pHolm'], 4))

## Side sweeps and studies (`--flags sweeps`)

In [ ]:
#| sweeps
t0 = time.time()
cells2 = sideSweeps('sweeps')
rows2 = runGrid(cells2, WORKERS)
wall2 = time.time() - t0
tab2 = sweepTable(rows2)
opt2, sb2 = teamOptimal(tab2), studyB(rows2)
meta2 = dict(mode='sweeps', cells=len(cells2), workers=WORKERS, wallSeconds=wall2, preregHash=preregHash(),
             gitCommit=gitCommit(), prereg=PREREG, deviations=DEVIATIONS)
path2 = save('sweeps', rows2, sweepFigures(tab2, opt2, sb2), meta2,
             dict(sweeps=jsonTable(tab2), teamOptimalCUCB=opt2, studyB=sb2),
             report(meta2, None, None, tab2, opt2, sb2))
print(f'sweeps: {len(rows2)} rows in {wall2:.0f} s ->', path2)
print('team-optimal cUCB:', opt2)